**historical stock analysis - practice set for additional credit (assignment ec1)**

this notebook works through exercises a to f from the ec1 practice set. it builds on eda.ipynb (assignment 2) and assignment3_analysis.ipynb (assignment 3), reusing the same dataset, groupings, and hypotheses rather than starting over.

- exercise a: redesign the hardest to read chart from assignment 2
- exercise b: small multiples across a category
- exercise c: interactive linked views
- exercise d: statistical rigor on one of our assignment 3 hypotheses
- exercise e: a reusable style module
- exercise f: the additional analysis flagged in assignment 3 (wider region basket), plus critique checklist

**exercise a: redesign for clarity**

a.1 pick the single hardest to read chart from assignment 2 or the assignment 3 prototype

we are picking eda.ipynb's figure 8, the chart that indexes six tickers (nvda, avgo, tsm, asml, aapl, msft) to 100 starting 2009-08-06. it looks fine at first glance since it already uses direct labels instead of a legend, but the y-axis is linear and the six lines end up 27 times apart by 2026, so most of the chart is not actually readable.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/stock_data.csv", parse_dates=["Date"])
df = df.sort_values(["Ticker", "Date"])
df["Daily Return"] = df.groupby("Ticker")["Adj Close"].pct_change()
df.head()

,Date,Ticker,Open,High,Low,Close,Adj Close,Volume,Daily Return
0,2006-01-02,000660.KS,36100.0,37650.0,35850.0,37600.0,27358.396484,13334311.0,NaN
1,2006-01-03,000660.KS,38100.0,38400.0,36850.0,38200.0,27794.970703,13815168.0,0.015958
2,2006-01-04,000660.KS,38250.0,39050.0,35300.0,35400.0,25757.650391,23064896.0,-0.073298
3,2006-01-05,000660.KS,35900.0,36000.0,33600.0,34750.0,25284.687500,21043236.0,-0.018362
4,2006-01-06,000660.KS,34950.0,35850.0,34850.0,35100.0,25539.355469,12557226.0,0.010072


In [ ]:
# the original chart, exactly as it was built in eda.ipynb, so we have a real "before"

chart_tickers = ["NVDA", "AVGO", "TSM", "ASML", "AAPL", "MSFT"]
prices_wide = df[df["Ticker"].isin(chart_tickers)].pivot(index="Date", columns="Ticker", values="Adj Close")

rebase_date = prices_wide.dropna().index.min()
indexed = prices_wide.loc[rebase_date:] / prices_wide.loc[rebase_date] * 100

colors = {
    "NVDA": "#4C72B0", "AVGO": "#55A868", "TSM": "#8172B2",
    "ASML": "#C44E52", "AAPL": "#937860", "MSFT": "#CCB974",
}

fig, ax = plt.subplots(figsize=(10, 6))
for ticker in chart_tickers:
    ax.plot(indexed.index, indexed[ticker], color=colors[ticker], linewidth=1.5)
    ax.text(indexed.index[-1], indexed[ticker].iloc[-1], f" {ticker}", color=colors[ticker], va="center")

ax.set_title(f"Figure 8 (original): price indexed to 100 at {rebase_date.date()}, linear y-axis")
ax.set_xlabel("Date")
ax.set_ylabel("Indexed price, 100 = value on rebase date")
plt.tight_layout()
plt.savefig("assignments/figures/ec1/ec1_a_before.png", dpi=150, bbox_inches="tight")
plt.show()

print(indexed.iloc[-1].sort_values(ascending=False))

a.2 three specific things wrong with it

- wrong scale for the data's range. by 2026 nvda ends at 63,121 and msft ends at 2,294, a 27 times spread. on a linear y-axis that forces asml, tsm, aapl, and msft to sit almost flat against the bottom of the chart for most of the 16 year span, so you cannot actually see their growth, only the last year or two of it.

- poor pre-attentive encoding of change. a linear axis makes equal vertical distance mean equal absolute change, not equal percent change. that is the wrong encoding for an indexed/growth chart, where the question is "how many times did this grow," not "how many index points did it gain." the chart makes nvda's growth look almost infinitely bigger than msft's, when in percent terms it's a large but bounded difference.

- label collision. asml (6,247), tsm (5,820), and aapl (5,387) end within about 15 percent of each other, so their direct end labels stack on top of each other at the right edge, exactly the kind of unclear labeling direct labels are supposed to fix.

In [ ]:
# a.3 redesign: same six lines, same data, but a log y-axis so every ticker's
# growth is visible across the whole 16 years, not just the last year or two.
# labels are also nudged apart when two tickers end up close together, so they
# do not print on top of each other.

final_values = indexed.iloc[-1].sort_values()

label_positions = {}
last_placed = 0
for ticker in final_values.index:
    value = final_values[ticker]
    if last_placed > 0 and value < last_placed * 1.15:
        placed = last_placed * 1.15
    else:
        placed = value
    label_positions[ticker] = placed
    last_placed = placed

fig, ax = plt.subplots(figsize=(10, 6))
for ticker in chart_tickers:
    ax.plot(indexed.index, indexed[ticker], color=colors[ticker], linewidth=1.5)
    ax.text(indexed.index[-1], label_positions[ticker], f" {ticker}", color=colors[ticker], va="center")

ax.set_yscale("log")
ax.set_title(f"Figure 8 (redesigned): price indexed to 100 at {rebase_date.date()}, log y-axis")
ax.set_xlabel("Date")
ax.set_ylabel("Indexed price, log scale, 100 = value on rebase date")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("assignments/figures/ec1/ec1_a_after.png", dpi=150, bbox_inches="tight")
plt.show()

a.4 before and after comparison

- before, only nvda and avgo are readable for most of the chart. asml, tsm, aapl, and msft look like a single flat line near zero until roughly 2023, even though all four grew several times over during that stretch

- after, switching to a log y-axis means equal vertical space represents equal percent change everywhere on the chart, so all six tickers show a visible growth curve across the full 2009 to 2026 span, not just the winners

- the redesign also nudges the three closest end labels (asml, tsm, aapl) apart by a small fixed amount so all six tickers are readable at the right edge, instead of three names overlapping into an unreadable stack

- nothing about the underlying data changed, only the scale and the label placement, which is the point: the first version was not wrong, it was just showing the reader six lines while making four of them invisible

**exercise b: small multiples storytelling**

b.1 categorical variable: region

assignment 3 used one ticker per region (jpm for us, 005930.ks for korea, 0700.hk for hong kong, novn.sw for switzerland, mc.pa for paris) to check h2 and h3, but it only ever plotted the five regions averaged into one bar per period. here we build one subplot per region instead, so we can see whether all five regions actually behave the same way, or whether the average was hiding something.

the relationship shown in each panel is the same one assignment 3's figure b already used: trading volume during each of the four periods (calm 2015-2017, 2008 crisis, 2020 covid crash, 2022 drawdown), divided by that ticker's own calm-period baseline. that keeps every panel on the same "times normal volume" unit even though the five tickers trade in five different currencies and share counts.

In [4]:
# b.2 set up: same region_ticker and periods dicts as assignment3_analysis.ipynb

region_ticker = {
    "US": "JPM",
    "Korea": "005930.KS",
    "Hong Kong": "0700.HK",
    "Switzerland": "NOVN.SW",
    "Paris": "MC.PA",
}

periods = {
    "calm\n2015-2017": ("2015-01-01", "2017-12-31"),
    "2008\ncrisis": ("2008-09-01", "2009-03-31"),
    "2020 covid\ncrash": ("2020-02-15", "2020-04-30"),
    "2022\ndrawdown": ("2022-01-01", "2022-12-31"),
}

tickers = list(region_ticker.values())
volume_wide = df[df["Ticker"].isin(tickers)].pivot(index="Date", columns="Ticker", values="Volume")

baseline_start, baseline_end = periods["calm\n2015-2017"]
baseline_volume = volume_wide.loc[baseline_start:baseline_end].mean()

# one volume ratio per region per period, kept separate instead of averaged together
region_period_ratio = {}
for region, ticker in region_ticker.items():
    ratios_for_region = []
    for period_name, (start, end) in periods.items():
        period_volume = volume_wide.loc[start:end, ticker].mean()
        ratios_for_region.append(period_volume / baseline_volume[ticker])
    region_period_ratio[region] = ratios_for_region

region_period_ratio

{'US': [np.float64(1.0),
  np.float64(5.109334751698659),
  np.float64(1.7569689861168531),
  np.float64(0.8752515700013412)],
 'Korea': [np.float64(1.0),
  np.float64(2.481443154749917),
  np.float64(2.272940059531812),
  np.float64(1.1752934214583928)],
 'Hong Kong': [np.float64(1.0),
  np.float64(1.545960381686968),
  np.float64(1.3197587802514414),
  np.float64(1.3589227286170948)],
 'Switzerland': [np.float64(1.0),
  np.float64(1.7925608875670245),
  np.float64(1.6550880528038092),
  np.float64(0.6943002621597684)],
 'Paris': [np.float64(1.0),
  np.float64(2.604922106991907),
  np.float64(1.3984834832160933),
  np.float64(0.5260515549671752)]}

In [ ]:
# b.3 one subplot per region, sharey=True so every panel uses the same y-axis range

period_labels = list(periods.keys())
bar_color = "#2a78d6"

fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=True)

for ax, region in zip(axes, region_ticker.keys()):
    ax.bar(period_labels, region_period_ratio[region], color=bar_color)
    ax.axhline(1.0, color="#898781", linestyle="--", linewidth=1)
    ax.set_title(region)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

axes[0].set_ylabel("Trading volume vs. calm baseline (times normal)")
fig.suptitle("Figure C: trading volume by region across four time periods, shared y-axis", fontsize=12)
plt.figtext(
    0.01, -0.05,
    "Source: data/stock_data.csv. One ticker per region: JPM (US), 005930.KS (Korea), 0700.HK (Hong Kong),\n"
    "NOVN.SW (Switzerland), MC.PA (Paris). Each bar is that period's average volume divided by the same\n"
    "ticker's 2015-2017 calm baseline average.",
    fontsize=8,
)
plt.tight_layout()
plt.savefig("assignments/figures/ec1/ec1_b_small_multiples.png", dpi=150, bbox_inches="tight")
plt.show()

b.4 what small multiples reveal that the combined chart hid

- in assignment 3's figure b, the 2008 bar was an average of all five regions, about 2.7 times normal volume. that number made 2008 look like a broad, roughly even panic across all regions, similar in shape to 2020

- split by region, that is not what happened. the us panel (jpm) spikes to about 5.1 times normal in 2008, clearly the highest bar anywhere in this whole figure, while korea, hong kong, switzerland, and paris all sit in the 1.5 to 2.6 range that same period, closer to their own 2020 numbers than to jpm's

- this matches the caveat assignment 3 already raised about jpm's own news (the bear stearns acquisition in march 2008) possibly drowning out the regional signal for 2008, and the small multiples view is what actually shows it, since averaging jpm in with four calmer regions pulls the combined number down and hides how much of an outlier the us region really was that period

**exercise c: interactive linked views**

c.1 two related charts: a scatter of daily return over time for the h1 tickers (chip/ai group and rest of tech group, since 2023-01-03), and a bar chart of each ticker's average daily return over whatever date range is currently selected in the scatter.

c.2 / c.3 the two views are linked with altair's selection_interval brush on the scatter's x-axis (date). dragging a date range in the scatter filters the bar chart to that same range, using transform_filter. on top of that brush there is a second control, a dropdown that isolates either the chip/ai group, the rest of tech group, or both, using a bind_radio selection on the group field.

In [6]:
import altair as alt

alt.data_transformers.disable_max_rows()

# same two groups assignment3_analysis.ipynb used for h1
chip_group = ["NVDA", "AVGO", "TSM", "ASML", "AMD", "MU", "LRCX"]
rest_group = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NFLX", "ORCL", "CSCO"]
h1_tickers = chip_group + rest_group

start_date = "2023-01-03"
h1_returns = df[(df["Ticker"].isin(h1_tickers)) & (df["Date"] >= start_date)].copy()
h1_returns = h1_returns.dropna(subset=["Daily Return"])

# a group label column, chip/ai or rest of tech, one row per ticker per day
group_labels = []
for ticker in h1_returns["Ticker"]:
    if ticker in chip_group:
        group_labels.append("chip / AI")
    else:
        group_labels.append("rest of tech")
h1_returns["Group"] = group_labels

h1_returns = h1_returns[["Date", "Ticker", "Group", "Daily Return"]]
h1_returns.head()

,Date,Ticker,Group,Daily Return
34305,2023-01-03,AAPL,rest of tech,-0.037405
34306,2023-01-04,AAPL,rest of tech,0.010314
34307,2023-01-05,AAPL,rest of tech,-0.010605
34308,2023-01-06,AAPL,rest of tech,0.036794
34309,2023-01-09,AAPL,rest of tech,0.004089


In [ ]:
# brush selection on the date axis, plus a dropdown radio control for group
brush = alt.selection_interval(encodings=["x"])
group_dropdown = alt.selection_point(
    fields=["Group"],
    bind=alt.binding_radio(
        options=[None, "chip / AI", "rest of tech"],
        labels=["All", "Chip / AI", "Rest of tech"],
        name="Group: ",
    ),
)

group_colors = alt.Scale(domain=["chip / AI", "rest of tech"], range=["#0072B2", "#E69F00"])

scatter = (
    alt.Chart(h1_returns)
    .mark_circle(size=20, opacity=0.5)
    .encode(
        x=alt.X("Date:T", title="Date"),
        y=alt.Y("Daily Return:Q", title="Daily return", axis=alt.Axis(format="%")),
        color=alt.Color("Group:N", scale=group_colors),
        tooltip=["Ticker:N", "Date:T", alt.Tooltip("Daily Return:Q", format=".2%")],
    )
    .transform_filter(group_dropdown)
    .add_params(brush, group_dropdown)
    .properties(width=600, height=300, title="Daily returns since 2023-01-03, drag to select a date range")
)

bars = (
    alt.Chart(h1_returns)
    .mark_bar()
    .encode(
        x=alt.X("Ticker:N", sort="-y", title="Ticker"),
        y=alt.Y("mean(Daily Return):Q", title="Average daily return, selected range", axis=alt.Axis(format="%")),
        color=alt.Color("Group:N", scale=group_colors),
        tooltip=[alt.Tooltip("mean(Daily Return):Q", format=".2%")],
    )
    .transform_filter(brush)
    .transform_filter(group_dropdown)
    .properties(width=600, height=250, title="Average daily return by ticker, over the brushed date range")
)

linked_view = scatter & bars
linked_view.save("assignments/figures/ec1/ec1_c_linked_views.html")
linked_view.save("assignments/figures/ec1/ec1_c_linked_views.png")
linked_view

c.4 what this makes possible

- a static version of this chart can only show one fixed window, the full 2023-01-03 to 2026-02-20 period, so it only supports one question: "over the whole period, which group did better." the ranked bars above already show that, chip/ai names take 4 of the top 5 spots

- with the brush, a viewer can drag over just the 2025 stretch, or just the couple of weeks around a specific selloff, and the bar chart instantly recomputes each ticker's average return for exactly that window. that turns "did the chip group win overall" into "did the chip group also win during this specific period," which the static chart cannot answer without redrawing it by hand

- the group dropdown answers a different kind of question, whether one group's own spread of names (for example is it just nvda carrying the whole chip group, or all seven) is doing the work, by hiding the other group's points and bars instead of making the reader mentally filter by color

- note: the image above only shows the default, unfiltered state, since a static png cannot capture dragging or clicking. assignments/figures/ec1/ec1_c_linked_views.html is the actual interactive version and should be opened in a browser to brush and filter it

**exercise d: statistical rigor**

d.1 hypothesis chosen: h1, the chip/ai rally is narrower than the rest of tech

figure a in assignment 3 showed this with cumulative indexed price, which is easy to read but can be misled by a handful of huge days compounding. here we check the same two groups a different way: the average single-day return, with a 95 percent confidence interval on each group's mean, plus a t-test on whether the two groups' daily returns actually differ.

In [8]:
import numpy as np
from scipy import stats

chip_returns = h1_returns[h1_returns["Group"] == "chip / AI"]["Daily Return"]
rest_returns = h1_returns[h1_returns["Group"] == "rest of tech"]["Daily Return"]

def mean_and_ci(returns):
    mean = returns.mean()
    standard_error = returns.std(ddof=1) / np.sqrt(len(returns))
    lower = mean - 1.96 * standard_error
    upper = mean + 1.96 * standard_error
    return mean, lower, upper

chip_mean, chip_lower, chip_upper = mean_and_ci(chip_returns)
rest_mean, rest_lower, rest_upper = mean_and_ci(rest_returns)

t_stat, p_value = stats.ttest_ind(chip_returns, rest_returns, equal_var=False)

print(f"chip / AI mean daily return: {chip_mean:.4%}, 95% CI [{chip_lower:.4%}, {chip_upper:.4%}]")
print(f"rest of tech mean daily return: {rest_mean:.4%}, 95% CI [{rest_lower:.4%}, {rest_upper:.4%}]")
print(f"welch's t-test: t = {t_stat:.2f}, p = {p_value:.4f}")

chip / AI mean daily return: 0.2636%, 95% CI [0.1861%, 0.3411%]
rest of tech mean daily return: 0.1374%, 95% CI [0.0875%, 0.1873%]
welch's t-test: t = 2.69, p = 0.0073


In [ ]:
group_names = ["chip / AI", "rest of tech"]
means = [chip_mean, rest_mean]
lower_errors = [chip_mean - chip_lower, rest_mean - rest_lower]
upper_errors = [chip_upper - chip_mean, rest_upper - rest_mean]
bar_colors = ["#0072B2", "#E69F00"]

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(group_names, means, color=bar_colors, yerr=[lower_errors, upper_errors], capsize=6)
ax.axhline(0, color="#898781", linewidth=1)
ax.set_title("Figure D: mean daily return by group since 2023-01-03, with 95% CI")
ax.set_ylabel("Mean daily return")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.2%}")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.text(0.5, 0.95, f"Welch's t-test: t = {t_stat:.2f}, p = {p_value:.4f}",
        transform=ax.transAxes, ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("assignments/figures/ec1/ec1_d_h1_stats.png", dpi=150, bbox_inches="tight")
plt.show()

d.2 / d.3 verdict

- the chip/ai group averages about 0.26% a day since 2023-01-03, versus about 0.14% for the rest of tech group, and a welch's t-test on the daily returns gives p = 0.0073, well under the usual 0.05 cutoff, so this is not just noise

- this supports h1 rather than complicating it. figure a's indexed price gap could in theory have come from just one or two huge days for the chip group, but this test looks at the average ordinary day instead and still finds a real difference

- it does add one honest complication: 0.26% vs 0.14% a day is a small gap on any single day, and the two group's 95% confidence intervals almost touch. the gap only turns into the dramatic split seen in figure a because it compounds over three years of trading days, not because any single day looks dramatically different between the two groups

**exercise e: reusable style module**

e.1 the style lives in scripts/chart_style.py: a fixed six color palette (okabe-ito, a standard colorblind safe set), the same two colors for the chip/ai and rest of tech groups everywhere they appear, standard font sizes, and a standard figure size. we ran it through the course's palette validator, which passed all the colorblind-safety checks.

e.2 below we re-render assignment 3's figure a (h1, chip/ai vs rest of tech) and figure b (h2/h3, correlation/volatility/volume across four periods) using this shared style, instead of the ad hoc colors those two charts picked independently.

In [ ]:
import sys
sys.path.insert(0, "scripts")
import chart_style

# same h1 groups and indexing assignment3_analysis.ipynb used for figure a
recent = df[df["Date"] >= "2023-01-03"]
prices = recent.pivot(index="Date", columns="Ticker", values="Adj Close")
indexed_h1 = prices / prices.iloc[0] * 100
chip_line = indexed_h1[chip_group].mean(axis=1)
rest_line = indexed_h1[rest_group].mean(axis=1)

fig, ax = plt.subplots(figsize=chart_style.FIGSIZE_STANDARD)
ax.plot(chip_line.index, chip_line.values, color=chart_style.GROUP_COLORS["chip / AI"], linewidth=2)
ax.plot(rest_line.index, rest_line.values, color=chart_style.GROUP_COLORS["rest of tech"], linewidth=2)
ax.text(chip_line.index[-1], chip_line.iloc[-1], "  chip / AI group", color=chart_style.GROUP_COLORS["chip / AI"], va="center")
ax.text(rest_line.index[-1], rest_line.iloc[-1], "  rest of tech group", color=chart_style.GROUP_COLORS["rest of tech"], va="center")

ax.set_title("Figure A (restyled): chip/AI vs rest of tech, indexed to 100 on 2023-01-03")
ax.set_xlabel("Date")
ax.set_ylabel("Indexed price, 100 = value on 2023-01-03")
chart_style.apply_style(ax)
plt.tight_layout()
plt.savefig("assignments/figures/ec1/ec1_e_figure_a_restyled.png", dpi=150, bbox_inches="tight")
plt.show()

In [11]:
# same correlation/volatility/volume setup as assignment3_analysis.ipynb's figure b,
# reusing the region_ticker and periods dicts from exercise b above

pairs = [
    ("JPM", "005930.KS"), ("JPM", "0700.HK"), ("JPM", "NOVN.SW"), ("JPM", "MC.PA"),
    ("005930.KS", "0700.HK"), ("005930.KS", "NOVN.SW"), ("005930.KS", "MC.PA"),
    ("0700.HK", "NOVN.SW"), ("0700.HK", "MC.PA"),
    ("NOVN.SW", "MC.PA"),
]

returns_wide = df[df["Ticker"].isin(tickers)].pivot(index="Date", columns="Ticker", values="Daily Return")

results = []
for period_name, (start, end) in periods.items():
    period_returns = returns_wide.loc[start:end]
    corr_matrix = period_returns.corr()

    pair_correlations = []
    for ticker_a, ticker_b in pairs:
        pair_correlations.append(corr_matrix.loc[ticker_a, ticker_b])
    avg_correlation = sum(pair_correlations) / len(pair_correlations)

    ticker_volatilities = []
    for ticker in tickers:
        ticker_volatilities.append(period_returns[ticker].std())
    avg_volatility = sum(ticker_volatilities) / len(ticker_volatilities)

    period_volume = volume_wide.loc[start:end].mean()
    volume_ratio_per_ticker = period_volume / baseline_volume
    avg_volume_ratio = volume_ratio_per_ticker.mean()

    results.append({
        "period": period_name.replace("\n", " "),
        "avg_correlation": avg_correlation,
        "avg_volatility": avg_volatility,
        "avg_volume_ratio": avg_volume_ratio,
    })

results_table = pd.DataFrame(results)
results_table

,period,avg_correlation,avg_volatility,avg_volume_ratio
0,calm 2015-2017,0.250209,0.015062,1.000000
1,2008 crisis,0.250385,0.045518,2.706844
2,2020 covid crash,0.526433,0.035688,1.680648
3,2022 drawdown,0.209277,0.019800,0.925964


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=chart_style.FIGSIZE_WIDE)
period_labels = list(periods.keys())
bar_color = chart_style.PALETTE[0]

axes[0].bar(period_labels, results_table["avg_correlation"], color=bar_color)
axes[0].set_title("Avg. correlation\nbetween regions")
axes[0].set_ylabel("Correlation (unitless)")

axes[1].bar(period_labels, results_table["avg_volatility"], color=bar_color)
axes[1].set_title("Avg. volatility\n(std of daily return)")
axes[1].set_ylabel("Volatility (unitless)")

axes[2].bar(period_labels, results_table["avg_volume_ratio"], color=bar_color)
axes[2].axhline(1.0, color="#898781", linestyle="--", linewidth=1)
axes[2].set_title("Avg. trading volume\n(vs. calm baseline)")
axes[2].set_ylabel("Times normal volume")

for ax in axes:
    chart_style.apply_style(ax)

fig.suptitle("Figure B (restyled): correlation, volatility, and volume across four time periods", fontsize=chart_style.FONT_SIZE_TITLE)
plt.tight_layout()
plt.savefig("assignments/figures/ec1/ec1_e_figure_b_restyled.png", dpi=150, bbox_inches="tight")
plt.show()

e.3 why these colors

- the palette is okabe-ito, a standard colorblind safe set, and we ran it through the course's validator script rather than eyeballing it. it passed the colorblind separation checks for every pair we use

- blue and orange are assigned to the same two groups (chip/ai and rest of tech) everywhere they appear in this notebook and will keep meaning the same thing in every chart we add later, so color follows the entity instead of being repicked per chart

- these are categorical groups, not a magnitude, so a categorical palette is the right encoding here rather than a single-hue gradient, which we would only reach for if we were shading one variable like volatility from low to high

**exercise f: finishing the additional analysis proposed in assignment 3**

f.1 assignment 3 flagged widening the region basket from one stock to two or three as its next step, specifically because jpm's own bear stearns news in 2008 looked like it was drowning out the regional signal that year. here we do that.

we kept every ticker that has full 2006 to 2026 history so all four periods stay comparable, and picked companies from different sectors within a region where the dataset allows it, so no single company's own news can dominate a region average the way jpm's did:

- us: jpm (finance), aapl (tech), xom (energy)
- korea: 005930.ks (samsung), 000660.ks (sk hynix). both are semiconductor/tech names, there is no non-tech korean ticker in this dataset, so this basket still leans tech
- hong kong: 0700.hk (tencent, tech), 0939.hk (china construction bank), 1398.hk (icbc)
- switzerland: novn.sw (novartis), rog.sw (roche). both pharma, same limitation as korea
- paris: mc.pa (lvmh) only. this is the one region we genuinely cannot widen, mc.pa is the only paris-listed ticker anywhere in the 49 ticker dataset. this stays a known limitation, not something this exercise fixes

In [13]:
import itertools

region_tickers = {
    "US": ["JPM", "AAPL", "XOM"],
    "Korea": ["005930.KS", "000660.KS"],
    "Hong Kong": ["0700.HK", "0939.HK", "1398.HK"],
    "Switzerland": ["NOVN.SW", "ROG.SW"],
    "Paris": ["MC.PA"],
}
all_region_tickers = []
for basket in region_tickers.values():
    all_region_tickers.extend(basket)

wide_returns = df[df["Ticker"].isin(all_region_tickers)].pivot(index="Date", columns="Ticker", values="Daily Return")
wide_volume = df[df["Ticker"].isin(all_region_tickers)].pivot(index="Date", columns="Ticker", values="Volume")

# one average return series per region, same idea as chip_line/rest_line in exercise d,
# just five groups instead of two
region_returns = pd.DataFrame()
for region, basket in region_tickers.items():
    region_returns[region] = wide_returns[basket].mean(axis=1)

region_returns.head()

,US,Korea,Hong Kong,Switzerland,Paris
Date,,,,,
2006-01-02,NaN,NaN,NaN,NaN,NaN
2006-01-03,NaN,0.011009,NaN,-0.004924,0.002631
2006-01-04,-0.000373,-0.011800,0.037972,0.010262,0.013779
2006-01-05,-0.003264,-0.022093,-0.016908,-0.000651,-0.001941
2006-01-06,0.017530,0.008718,0.036862,0.000775,0.000000


In [14]:
# same four periods and same three metrics as assignment 3's figure b, just run
# on the widened region baskets instead of one ticker per region

region_pairs = list(itertools.combinations(region_tickers.keys(), 2))
wide_baseline_volume = wide_volume.loc[baseline_start:baseline_end].mean()

wide_results = []
for period_name, (start, end) in periods.items():
    period_region_returns = region_returns.loc[start:end]
    region_corr = period_region_returns.corr()

    pair_correlations = []
    for region_a, region_b in region_pairs:
        pair_correlations.append(region_corr.loc[region_a, region_b])
    avg_correlation = sum(pair_correlations) / len(pair_correlations)

    avg_volatility = period_region_returns.std().mean()

    period_volume = wide_volume.loc[start:end].mean()
    volume_ratio_per_ticker = period_volume / wide_baseline_volume
    region_ratios = []
    for region, basket in region_tickers.items():
        region_ratios.append(volume_ratio_per_ticker[basket].mean())
    avg_volume_ratio = sum(region_ratios) / len(region_ratios)

    wide_results.append({
        "period": period_name.replace("\n", " "),
        "avg_correlation": avg_correlation,
        "avg_volatility": avg_volatility,
        "avg_volume_ratio": avg_volume_ratio,
    })

wide_results_table = pd.DataFrame(wide_results)
wide_results_table

,period,avg_correlation,avg_volatility,avg_volume_ratio
0,calm 2015-2017,0.281787,0.013436,1.000000
1,2008 crisis,0.326846,0.040273,2.990198
2,2020 covid crash,0.538682,0.032838,1.738362
3,2022 drawdown,0.195714,0.016120,0.871323


In [ ]:
# figure f: does widening the basket actually change the 2008 correlation reading,
# grouped bars, single stock per region (assignment 3) vs two-three per region (here)

x = range(len(period_labels))
bar_width = 0.35
single_color = chart_style.PALETTE[0]
basket_color = chart_style.PALETTE[1]

fig, ax = plt.subplots(figsize=chart_style.FIGSIZE_WIDE)
ax.bar([i - bar_width / 2 for i in x], results_table["avg_correlation"], width=bar_width,
       color=single_color, label="one stock per region (assignment 3)")
ax.bar([i + bar_width / 2 for i in x], wide_results_table["avg_correlation"], width=bar_width,
       color=basket_color, label="2-3 stocks per region (this exercise)")

ax.set_xticks(list(x))
ax.set_xticklabels(period_labels)
ax.set_ylabel("Avg. correlation between regions")
ax.set_title("Figure F: widening the region basket changes the 2008 reading")
ax.legend(frameon=False)
chart_style.apply_style(ax)
plt.tight_layout()
plt.savefig("assignments/figures/ec1/ec1_f_widened_basket.png", dpi=150, bbox_inches="tight")
plt.show()

print("single stock, 2008 vs calm:", round(results_table.loc[1, "avg_correlation"], 3), "vs", round(results_table.loc[0, "avg_correlation"], 3))
print("widened basket, 2008 vs calm:", round(wide_results_table.loc[1, "avg_correlation"], 3), "vs", round(wide_results_table.loc[0, "avg_correlation"], 3))

interpretation

- with one stock per region, 2008's average correlation (0.250) was barely different from the calm baseline (0.250), which assignment 3 already flagged as suspicious given how much of a crisis 2008 actually was

- with two to three stocks per region, 2008 moves to 0.326 while the calm baseline only moves slightly to 0.282, so 2008 now sits clearly above calm instead of sitting right on top of it. this is exactly the cleanup assignment 3's next step predicted: averaging jpm in with aapl and xom dilutes jpm's own bear stearns driven trading, so the region's return series looks less like "one bank's crisis" and more like a broad us market move that actually does correlate more with other regions during a global panic

- 2020 and 2022 barely move (0.527 to 0.539, and 0.209 to 0.196), which is a useful check in itself: the single stock per region approach was not wrong everywhere, it specifically undersold 2008, which is the one period where the original single ticker happened to have a large company specific news event on top of the broader crisis

- this still does not fully fix h2's evidence for 2008, since the korea and switzerland baskets are still sector-narrow (both tech, both pharma) and paris still cannot be widened at all with this dataset, so 2008 for those three regions is still resting on thinner ground than the us and hong kong regions

f.3 critique checklist, self review of figure f

- is the chart type appropriate for this data and question? yes, 5/5. grouped bars are the right type for comparing two versions of the same measurement across four discrete periods, nothing here is continuous

- is there unnecessary chart junk? no, 5/5. no gridlines, no 3d, no border on top or right, only one reference detail (the legend) and it is necessary since there are two bars per period. one improvement: the four period labels could use a shared date subtitle instead of repeating "2015-2017" etc, but that is minor

- are axes, labels, and legends clear without needing explanation? yes, 4/5. one improvement: "avg. correlation between regions" does not say correlation of what (daily returns), a reader outside the team would need the caption to know that

- is color used meaningfully, not just decoratively? yes, 5/5. blue and orange separate the two methods being compared, not five arbitrary regions, and the same two colors mean the same two things everywhere else in this notebook

- does the written interpretation match what the chart actually shows? yes, 5/5, the interpretation calls out the 2008 gap specifically and also honestly reports that 2020 and 2022 barely moved, instead of only reporting the result that supports the point

peer half of f.3, giving and receiving feedback with another team's figure f, still needs to happen with an actual second team and cannot be filled in here. the checklist table in the report is left blank for that exchange.